In [1]:
!pip install -q timm
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score, f1_score
from collections import Counter

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [3]:
DATA_ROOT = Path("/kaggle/input/datasets/naumisharanyatirth/human-embryo-dataset")
IMG_ROOT  = DATA_ROOT / "embryo_dataset"
ANN_ROOT  = DATA_ROOT / "embryo_dataset_annotations"

BATCH_SIZE = 1024
EPOCH_STAGE1 = 1
EPOCH_STAGE2 = 1
NUM_CLASSES = 16

In [4]:
STAGES = [
    "tPB2","tPNa","tPNf","t2","t3","t4","t5","t6",
    "t7","t8","t9+","tM","tSB","tB","tEB","tHB"
]
LABEL_MAP = {stage:i for i, stage in enumerate(STAGES)}

In [5]:
def load_dataset():
    samples = []

    for csv_path in ANN_ROOT.glob("*_phases.csv"):
        patient = csv_path.stem.replace("_phases","")
        img_folder = IMG_ROOT / patient

        if not img_folder.exists():
            continue

        df = pd.read_csv(csv_path, header=None, names=["stage","start","end"])

        frame_map = {}
        for img in img_folder.glob("*.jpeg"):
            run = img.stem.split("RUN")[-1]
            if run.isdigit():
                frame_map[int(run)] = img

        for _, row in df.iterrows():
            stage = str(row["stage"]).strip()
            if stage not in LABEL_MAP:
                continue

            for f in range(int(row["start"]), int(row["end"])+1):
                if f in frame_map:
                    samples.append((str(frame_map[f]), LABEL_MAP[stage], patient))

    print("Total samples:", len(samples))
    return samples

samples = load_dataset()

Total samples: 297428


In [6]:
X = [s[0] for s in samples]
y = [s[1] for s in samples]
groups = [s[2] for s in samples]

gss = GroupShuffleSplit(test_size=0.2, n_splits=1)
train_idx, temp_idx = next(gss.split(X,y,groups))

X_train = [X[i] for i in train_idx]
y_train = [y[i] for i in train_idx]

X_temp = [X[i] for i in temp_idx]
y_temp = [y[i] for i in temp_idx]
groups_temp = [groups[i] for i in temp_idx]

gss2 = GroupShuffleSplit(test_size=0.5, n_splits=1)
val_idx, test_idx = next(gss2.split(X_temp,y_temp,groups_temp))

X_val = [X_temp[i] for i in val_idx]
y_val = [y_temp[i] for i in val_idx]

X_test = [X_temp[i] for i in test_idx]
y_test = [y_temp[i] for i in test_idx]

**AUGMENTAION**

In [7]:
train_tf = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(0.2,0.2),
    transforms.ToTensor(),
])

val_tf = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
])

**DATASET CLASS**

In [8]:
class EmbryoDataset(Dataset):
    def __init__(self, X, y, tf):
        self.X = X
        self.y = y
        self.tf = tf

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        img = Image.open(self.X[idx]).convert("RGB")
        img = self.tf(img)
        label = self.y[idx]
        return img, label

train_loader = DataLoader(
    EmbryoDataset(X_train, y_train, train_tf),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

val_loader = DataLoader(
    EmbryoDataset(X_val, y_val, val_tf),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

test_loader = DataLoader(
    EmbryoDataset(X_test, y_test, val_tf),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

**HYBRID LOSS**

In [9]:
class HybridLoss(nn.Module):
    def __init__(self, beta=0.3):
        super().__init__()
        self.ce = nn.CrossEntropyLoss()
        self.beta = beta

    def forward(self, logits, targets):
        ce = self.ce(logits, targets)

        probs = torch.softmax(logits, dim=1)
        idx = torch.arange(NUM_CLASSES).float().to(logits.device)

        dist = torch.abs(idx.unsqueeze(0) - targets.unsqueeze(1))
        ordinal = (probs * torch.log1p(dist)).sum(dim=1).mean()

        return ce + self.beta * ordinal


In [10]:
def get_model(name):
    if name=="mobilenet":
        m = models.mobilenet_v2(weights="IMAGENET1K_V1")
        m.classifier[1] = nn.Linear(m.last_channel, NUM_CLASSES)

    elif name=="vgg16":
        m = models.vgg16(weights="IMAGENET1K_V1")
        m.classifier[6] = nn.Linear(4096, NUM_CLASSES)

    elif name=="vgg19":
        m = models.vgg19(weights="IMAGENET1K_V1")
        m.classifier[6] = nn.Linear(4096, NUM_CLASSES)

    elif name=="inception":
        m = models.inception_v3(weights="IMAGENET1K_V1")
        m.aux_logits=False
        m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES)

    m = m.to(device)

    if torch.cuda.device_count() > 1:
        print("Using", torch.cuda.device_count(), "GPUs")
        m = nn.DataParallel(m)
    
    return m

**MODEL TRAINING**

In [11]:
from tqdm import tqdm

In [12]:
def train_model(model, name):
    print(f"\nTraining {name}")

    # Stage 1: Freeze
    for p in model.parameters():
        p.requires_grad = False
    
    base_model = model.module if isinstance(model, nn.DataParallel) else model
    
    if hasattr(base_model, "classifier"):
        for p in base_model.classifier.parameters():
            p.requires_grad = True
    elif hasattr(base_model, "fc"):
        for p in base_model.fc.parameters():
            p.requires_grad = True

    optimizer = torch.optim.Adam(filter(lambda p:p.requires_grad, model.parameters()), lr=1e-3)
    ce_loss = nn.CrossEntropyLoss()

    for epoch in range(EPOCH_STAGE1):
        model.train()
        total = 0
    
        loop = tqdm(train_loader, desc=f"Stage1 Epoch {epoch+1}/{EPOCH_STAGE1}")
    
        for x, y in loop:
            x, y = x.to(device), y.to(device)
    
            optimizer.zero_grad()
            out = model(x)
            if isinstance(out, tuple): out = out[0]
    
            loss = ce_loss(out, y)
            loss.backward()
            optimizer.step()
    
            total += loss.item()
    
            # live update
            loop.set_postfix(loss=loss.item())
    
        print(f"Stage1 Epoch {epoch+1}: Avg Loss={total/len(train_loader):.4f}")
    
    # Stage 2: Unfreeze
    for p in model.parameters():
        p.requires_grad=True

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    criterion = HybridLoss()

    for epoch in range(EPOCH_STAGE2):
        model.train()
        total = 0
        
        loop = tqdm(train_loader, desc=f"Stage2 Epoch {epoch+1}/{EPOCH_STAGE2}")
        
        for x, y in loop:
            x, y = x.to(device), y.to(device)
        
            optimizer.zero_grad()
            out = model(x)
            if isinstance(out, tuple): out = out[0]
        
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
        
            total += loss.item()
        
            # live update
            loop.set_postfix(loss=loss.item())
        
        print(f"Stage2 Epoch {epoch+1}: Avg Loss={total/len(train_loader):.4f}")
    return model


**MODELS**

In [13]:

import gc
gc.collect()

import torch
torch.cuda.empty_cache()

In [ ]:
models_dict = {}

for name in ["mobilenet", "vgg16", "vgg19", "inception"]:
    model = get_model(name)
    model = train_model(model, name)
    models_dict[name] = model


Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 231MB/s]


Using 2 GPUs

Training mobilenet


Stage1 Epoch 1/1:  64%|██████▍   | 148/232 [11:28<03:43,  2.66s/it, loss=1.57]

**EVALUATION**

In [ ]:
def evaluate(model):
    model.eval()
    y_true, y_pred = [], []

    with torch.no_grad():
        for x,y in test_loader:
            x = x.to(device)

            out = model(x)
            if isinstance(out,tuple): out=out[0]

            preds = out.argmax(dim=1).cpu().numpy()

            y_pred.extend(preds)
            y_true.extend(y.numpy())

    acc = accuracy_score(y_true,y_pred)
    f1  = f1_score(y_true,y_pred, average="weighted")

    return acc, f1

**RESULTS**

In [ ]:
results = {}

for name, model in models_dict.items():
    acc, f1 = evaluate(model)
    results[name] = (acc,f1)
    print(f"{name}: ACC={acc:.4f}, F1={f1:.4f}")

**VISUALIZATION**

In [ ]:
names = list(results.keys())
accs = [results[n][0] for n in names]

plt.figure()
plt.bar(names, accs)
plt.title("Model Accuracy Comparison")
plt.show()